# Poder discriminante de la entropía de von Neumann en árboles

Se analizan las bases SQLite de árboles no isomorfos para los órdenes $2\le n\le23$. El estudio compara los espectros laplacianos, las entropías numéricas y la capacidad complementaria de los índices de Zagreb y Wiener.

## 1. Configuración

Las bases deben denominarse `arboles_orden_n.sqlite` y contener `indice`, `codigo`, `entropia`, `zagreb1` y `wiener`. La búsqueda incluye subcarpetas.

In [ ]:
import csv
import json
import random
import sqlite3
from collections import Counter, defaultdict
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N = 23
N_MIN = 2
EPS_ENTROPIA = 1e-10
EPS_ESPECTRO = 1e-10
MAX_PUNTOS_GRAFICO = 50000
MUESTRA_SEMILLA = 2026

CARPETA_BASES = Path.cwd()
CARPETA_RESULTADOS = Path(f"analisis_orden_{N}")
CARPETA_TABLAS = CARPETA_RESULTADOS / "tablas"
CARPETA_IMAGENES = CARPETA_RESULTADOS / "imagenes"
CARPETA_CACHE = CARPETA_RESULTADOS / "cache"

for carpeta in (CARPETA_TABLAS, CARPETA_IMAGENES, CARPETA_CACHE):
    carpeta.mkdir(parents=True, exist_ok=True)

# Las muestras gráficas son aleatorias y reproducibles.
# Los resultados numéricos se calculan con todos los registros.

## 2. Lectura de las bases

Solo se emplean bases completas. La validación se limita al esquema y a la cantidad esperada de árboles.

In [ ]:
COLUMNAS = {"indice", "codigo", "entropia", "zagreb1", "wiener"}


def localizar_bases():
    encontrados = {}
    for ruta in CARPETA_BASES.rglob("arboles_orden_*.sqlite"):
        try:
            n = int(ruta.stem.removeprefix("arboles_orden_"))
        except ValueError:
            continue
        if N_MIN <= n <= N:
            if n in encontrados:
                raise ValueError(f"Hay más de una base para n={n}.")
            encontrados[n] = ruta.resolve()

    faltantes = [n for n in range(N_MIN, N + 1) if n not in encontrados]
    if faltantes:
        raise FileNotFoundError(f"Faltan las bases de los órdenes: {faltantes}")
    return encontrados


def abrir_base(n, ruta):
    conexion = sqlite3.connect(ruta.as_uri() + "?mode=ro", uri=True)
    columnas = {
        fila[1] for fila in conexion.execute("PRAGMA table_info(arboles)")
    }
    if not COLUMNAS.issubset(columnas):
        conexion.close()
        raise ValueError(f"Columnas insuficientes en la base de n={n}.")

    cantidad = conexion.execute("SELECT COUNT(*) FROM arboles").fetchone()[0]
    esperado = nx.number_of_nonisomorphic_trees(n)
    if cantidad != esperado:
        conexion.close()
        raise ValueError(f"Base incompleta de n={n}: {cantidad:,}/{esperado:,} árboles.")
    return conexion, cantidad


bases = localizar_bases()
print(f"Bases disponibles: órdenes {N_MIN} a {N}.")

## 3. Comparación espectral

Los árboles se agrupan en intervalos numéricos de entropía cuya amplitud no excede $\varepsilon=10^{-10}$. Los espectros se comparan con tolerancia $	au=10^{-10}$. Los índices $Z_1$ y $W$ permiten descartar cospectralidad cuando son diferentes.

In [ ]:
def espectro_laplaciano(codigo):
    T = nx.from_graph6_bytes(codigo.encode("ascii"))
    n = T.number_of_nodes()
    L = np.zeros((n, n), dtype=float)
    for v, d in T.degree():
        L[v, v] = d
    for u, v in T.edges():
        L[u, v] = L[v, u] = -1.0
    return np.linalg.eigvalsh(L)


def representantes_espectrales(grupo):
    """Devuelve un representante por espectro numérico distinto."""
    por_indices = defaultdict(list)
    for r in grupo:
        por_indices[(r["zagreb1"], r["wiener"])].append(r)

    representantes = []
    for registros in por_indices.values():
        if len(registros) == 1:
            representantes.append(registros[0])
            continue

        espectros = []
        for r in registros:
            valores = espectro_laplaciano(r["codigo"])
            if not any(np.allclose(valores, ev, atol=EPS_ESPECTRO, rtol=0)
                       for ev in espectros):
                espectros.append(valores)
                representantes.append(r)
    return representantes


def pares_iguales(contador):
    return sum(c * (c - 1) // 2 for c in contador.values())

## 4. Análisis por orden

Para cada orden se obtienen el número de árboles $N_n$, el número de espectros distintos $Q_n$ y el número de grupos numéricos de entropía $E_n$. En los grupos con espectros distintos se contabilizan los pares separados por $Z_1$, por $W$ y por la pareja $(Z_1,W)$.

In [ ]:
def analizar_orden(n, ruta):
    conexion, total = abrir_base(n, ruta)
    n_entropias = 0
    n_espectros = 0
    clases_ce = 0
    pares_espectrales = 0
    pares_zagreb = 0
    pares_wiener = 0
    pares_ambos = 0
    registros_ce = []
    diferencias = []
    muestra = []
    rng = random.Random(MUESTRA_SEMILLA)
    leidos = 0

    def procesar_grupo(grupo):
        nonlocal n_entropias, n_espectros, clases_ce
        nonlocal pares_espectrales, pares_zagreb, pares_wiener, pares_ambos
        if not grupo:
            return
        n_entropias += 1
        reps = representantes_espectrales(grupo) if len(grupo) > 1 else grupo
        q = len(reps)
        n_espectros += q
        if q < 2:
            return

        clases_ce += 1
        total_pares = q * (q - 1) // 2
        iguales_z = pares_iguales(Counter(r["zagreb1"] for r in reps))
        iguales_w = pares_iguales(Counter(r["wiener"] for r in reps))
        iguales_zw = pares_iguales(Counter((r["zagreb1"], r["wiener"]) for r in reps))

        pares_espectrales += total_pares
        pares_zagreb += total_pares - iguales_z
        pares_wiener += total_pares - iguales_w
        pares_ambos += total_pares - iguales_zw

        if n == N:
            registros_ce.append({
                "clase_entropia": n_entropias,
                "n_arboles": len(grupo),
                "n_espectros": q,
                "S_min": min(r["entropia"] for r in grupo),
                "S_max": max(r["entropia"] for r in grupo),
                "Z1_min": min(r["zagreb1"] for r in reps),
                "Z1_max": max(r["zagreb1"] for r in reps),
                "W_min": min(r["wiener"] for r in reps),
                "W_max": max(r["wiener"] for r in reps),
                "pares_espectrales": total_pares,
                "separados_Z1": total_pares - iguales_z,
                "separados_W": total_pares - iguales_w,
                "separados_Z1_W": total_pares - iguales_zw,
            })
        if n == 20:
            for i in range(q):
                for j in range(i + 1, q):
                    diferencias.append(abs(reps[i]["entropia"] - reps[j]["entropia"]))

    grupo = []
    inicio_grupo = None
    try:
        cursor = conexion.execute(
            "SELECT indice, codigo, entropia, zagreb1, wiener "
            "FROM arboles ORDER BY entropia, indice"
        )
        for indice, codigo, S, z, w in cursor:
            r = {"indice": int(indice), "codigo": codigo,
                 "entropia": float(S), "zagreb1": int(z), "wiener": int(w)}
            leidos += 1
            if n == N:
                # Muestreo por reservorio, sin modificar el criterio aleatorio.
                punto = {k: r[k] for k in ("indice", "entropia", "zagreb1", "wiener")}
                if len(muestra) < MAX_PUNTOS_GRAFICO:
                    muestra.append(punto)
                else:
                    j = rng.randrange(leidos)
                    if j < MAX_PUNTOS_GRAFICO:
                        muestra[j] = punto

            if not grupo:
                grupo = [r]
                inicio_grupo = r["entropia"]
            elif r["entropia"] - inicio_grupo <= EPS_ENTROPIA:
                grupo.append(r)
            else:
                procesar_grupo(grupo)
                grupo = [r]
                inicio_grupo = r["entropia"]
        procesar_grupo(grupo)
    finally:
        conexion.close()

    if leidos != total:
        raise RuntimeError(f"Lectura incompleta para n={n}.")

    resumen = {
        "n": n, "n_arboles": total,
        "n_espectros_distintos": n_espectros,
        "n_clases_S": n_entropias,
        "perdida_arbol_espectro": total - n_espectros,
        "perdida_espectro_entropia": n_espectros - n_entropias,
        "n_clases_CE_no_cospectrales": clases_ce,
        "pares_espectrales_CE": pares_espectrales,
        "pares_separados_Z1": pares_zagreb,
        "pares_separados_W": pares_wiener,
        "pares_separados_Z1_W": pares_ambos,
        "pct_pares_separados_Z1": 100 * pares_zagreb / pares_espectrales if pares_espectrales else np.nan,
        "pct_pares_separados_W": 100 * pares_wiener / pares_espectrales if pares_espectrales else np.nan,
        "pct_pares_separados_Z1_W": 100 * pares_ambos / pares_espectrales if pares_espectrales else np.nan,
    }
    return resumen, registros_ce, diferencias, muestra

## 5. Evolución y almacenamiento

Se reutilizan los resultados de cada orden cuando la base no ha cambiado. Se conservan los datos de detalle de $n=23$, la muestra gráfica y las diferencias de entropía de $n=20$.

In [ ]:
def ruta_cache(n):
    return CARPETA_CACHE / f"resumen_n{n}.json"


def leer_cache(n, ruta):
    archivo = ruta_cache(n)
    if not archivo.exists():
        return None
    datos = json.loads(archivo.read_text(encoding="utf-8"))
    estado = [str(ruta), ruta.stat().st_size, ruta.stat().st_mtime_ns,
              EPS_ENTROPIA, EPS_ESPECTRO, MAX_PUNTOS_GRAFICO, MUESTRA_SEMILLA]
    if datos.get("estado") != estado:
        return None
    necesarios = []
    if n == N:
        necesarios = [CARPETA_TABLAS / "Clases_CE_n23.csv", CARPETA_TABLAS / "Muestra_graficos.csv"]
    elif n == 20 and N >= 20:
        necesarios = [CARPETA_TABLAS / "Diferencias_entropia_n20.csv"]
    if any(not archivo.exists() for archivo in necesarios):
        return None
    return datos["resumen"]


def guardar_cache(n, ruta, resumen):
    estado = [str(ruta), ruta.stat().st_size, ruta.stat().st_mtime_ns,
              EPS_ENTROPIA, EPS_ESPECTRO, MAX_PUNTOS_GRAFICO, MUESTRA_SEMILLA]
    ruta_cache(n).write_text(json.dumps({"estado": estado, "resumen": resumen},
                                     ensure_ascii=False, indent=2), encoding="utf-8")


filas = []
for n in range(N_MIN, N + 1):
    ruta = bases[n]
    resumen = leer_cache(n, ruta)
    if resumen is None:
        print(f"Análisis de árboles de orden {n}...")
        resumen, clases, diferencias, muestra = analizar_orden(n, ruta)
        if n == N:
            pd.DataFrame(clases).to_csv(CARPETA_TABLAS / "Clases_CE_n23.csv", index=False)
            pd.DataFrame(muestra).to_csv(CARPETA_TABLAS / "Muestra_graficos.csv", index=False)
        if n == 20:
            pd.DataFrame({"diferencia": diferencias}).to_csv(
                CARPETA_TABLAS / "Diferencias_entropia_n20.csv", index=False
            )
        guardar_cache(n, ruta, resumen)
    else:
        print(f"n={n}: resumen reutilizado.")
    filas.append(resumen)

tendencia_df = pd.DataFrame(filas)
tendencia_df.to_csv(CARPETA_TABLAS / "Tendencia_ordenes.csv", index=False)
print(tendencia_df[["n", "n_arboles", "n_espectros_distintos", "n_clases_S"]].to_string(index=False))

## 6. Figuras del informe

Se elaboran la comparación de pares para $n=23$, la evolución para $17\le n\le23$, la dispersión entropía–Zagreb mediante muestra aleatoria y el histograma de diferencias de entropía para $n=20$.

In [ ]:
def guardar_figura(fig, nombre):
    fig.tight_layout()
    fig.savefig(CARPETA_IMAGENES / nombre, dpi=300, bbox_inches="tight")
    plt.close(fig)


r = tendencia_df.loc[tendencia_df["n"] == N].iloc[0]

# Comparación de pares (n=23).
fig, ax = plt.subplots(figsize=(8, 5))
etiquetas = ["Zagreb", "Wiener", "(Zagreb, Wiener)"]
valores = [r["pct_pares_separados_Z1"], r["pct_pares_separados_W"],
           r["pct_pares_separados_Z1_W"]]
ax.barh(etiquetas, valores)
ax.set_xlabel("Pares espectrales separados (%)")
ax.set_title(f"Separación de pares coentrópicos no cospectrales, n={N}")
ax.set_xlim(0, 100)
guardar_figura(fig, f"comparacion_pares_n{N}.pdf")

# Evolución con n.
sub = tendencia_df[tendencia_df["n"].between(17, N)]
fig, ax = plt.subplots(figsize=(8, 5))
for columna, etiqueta in [
    ("pct_pares_separados_Z1", "Zagreb"),
    ("pct_pares_separados_W", "Wiener"),
    ("pct_pares_separados_Z1_W", "(Zagreb, Wiener)"),
]:
    ax.plot(sub["n"], sub[columna], marker="o", label=etiqueta)
ax.set_xlabel("Orden n")
ax.set_ylabel("Pares espectrales separados (%)")
ax.legend()
guardar_figura(fig, "evolucion_pares.pdf")

# Dispersión con el muestreo aleatorio establecido.
muestra_df = pd.read_csv(CARPETA_TABLAS / "Muestra_graficos.csv")
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(muestra_df["zagreb1"], muestra_df["entropia"], s=7, alpha=0.35)
ax.set_xlabel("Primer índice de Zagreb")
ax.set_ylabel("Entropía de von Neumann")
ax.set_title(f"Entropía y primer índice de Zagreb, n={N}")
guardar_figura(fig, "01_entropia_vs_zagreb.png")

# Diferencias de entropía en clases con espectros distintos (n=20).
if N >= 20:
    diferencias_df = pd.read_csv(CARPETA_TABLAS / "Diferencias_entropia_n20.csv")
    diferencias_positivas = diferencias_df.loc[
        diferencias_df["diferencia"] > 0, "diferencia"
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    if not diferencias_positivas.empty:
        ax.hist(np.log10(diferencias_positivas), bins=30)
    ax.set_xlabel(r"$\log_{10}(\Delta S)$")
    ax.set_ylabel("Número de pares")
    ax.set_title("Diferencias de entropía en pares no cospectrales, n=20")
    guardar_figura(fig, "histograma_delta_entropia.png")

## 7. Tablas de resultados

Se exportan los resúmenes de todos los órdenes, los resultados del orden principal y los archivos complementarios de las figuras.

In [ ]:
principal = tendencia_df.loc[tendencia_df["n"] == N].copy()
comparacion = pd.DataFrame({
    "descriptor": ["Zagreb", "Wiener", "(Zagreb, Wiener)"],
    "pares_separados": [int(r["pares_separados_Z1"]),
                        int(r["pares_separados_W"]),
                        int(r["pares_separados_Z1_W"])],
    "porcentaje": [r["pct_pares_separados_Z1"],
                   r["pct_pares_separados_W"],
                   r["pct_pares_separados_Z1_W"]],
})
comparacion.to_csv(CARPETA_TABLAS / "Comparacion_pares_n23.csv", index=False)
principal.to_csv(CARPETA_TABLAS / "Resumen_n23.csv", index=False)

with pd.ExcelWriter(CARPETA_TABLAS / "Resumen_seminario.xlsx") as writer:
    tendencia_df.to_excel(writer, sheet_name="Evolucion", index=False)
    principal.to_excel(writer, sheet_name="Orden_23", index=False)
    comparacion.to_excel(writer, sheet_name="Comparacion_pares", index=False)

print(f"Tablas: {CARPETA_TABLAS.resolve()}")
print(f"Figuras: {CARPETA_IMAGENES.resolve()}")